In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.dataset import get_transforms, make_loaders
from src.evaluate import get_logits, predict_loader, summarize
from src.experiment import run_experiment, run_experiment_tl
from src.model import CNNBaseline, ResNetFromScratch, TLResNet, describe_run
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()
paths = {
    'data': '../data/processed',
    'ckpt_dir': '../checkpoints',
    'log_dir': '../logs',
    'results': '../results/experiments.csv',
}

In [7]:

exp  = pd.read_csv('../results/experiments.csv')
bal  = pd.read_csv('../results/balanced.csv')
loss = pd.read_csv('../results/loss_comparison.csv')

SOURCES = [
    ('CNN baseline',                    exp,  'baseline_v2'),
    ('Balanced batches',                bal,  'imb_balanced_v2'),
    ('CrossEntropy / BCE comparison',   loss, 'bce_v2'),
    ('Best regularized + scheduled',    exp,  'best_conbo_v2'),
    ('ResNet feature extraction',       exp,  'resnet_fe'),
    ('ResNet fine-tuning',              exp,  'resnet_ft'),
]

cols = ['macro_precision', 'macro_recall', 'macro_f1',
        'lowest_recall_class', 'lowest_precision_class']

rows = []
for label, df, name in SOURCES:
    match = df[df['experiment'] == name]
    if len(match) == 0:
        print(f"⚠️ پیدا نشد: {name}")
        continue
    r = match.iloc[0]
    rows.append({'Experiment': label, **{c: r[c] for c in cols}})

table = pd.DataFrame(rows)
table[cols[:3]] = table[cols[:3]].round(3)
table.to_csv('../results/comparison_table.csv', index=False)

print('| ' + ' | '.join(table.columns) + ' |')
print('|' + '---|' * len(table.columns))
for _, r in table.iterrows():
    print('| ' + ' | '.join(str(v) for v in r) + ' |')

| Experiment | macro_precision | macro_recall | macro_f1 | lowest_recall_class | lowest_precision_class |
|---|---|---|---|---|---|
| CNN baseline | 0.612 | 0.6 | 0.588 | vanet | vanet |
| Balanced batches | 0.532 | 0.54 | 0.534 | vanet | vanet |
| CrossEntropy / BCE comparison | 0.49 | 0.538 | 0.501 | vanet | vanet |
| Best regularized + scheduled | 0.478 | 0.516 | 0.484 | vanet | vanet |
| ResNet feature extraction | 0.902 | 0.895 | 0.898 | vanet | vanet |
| ResNet fine-tuning | 0.936 | 0.93 | 0.932 | vanet | vanet |
